# Baseline Models for Daily Rat Sightings in New York City

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from pandas.tseries.holiday import USFederalHolidayCalendar


## Importing the Data

In [2]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Baseline Seasonal Average Model

In [3]:
years_back_use = 4
day_window_use = 4

In [4]:
def seasonal_average_forecast(data, target_dates, years_back=years_back_use, day_window=day_window_use):
    df = data.copy()
    df["ds"] = pd.to_datetime(df["ds"])
    df["doy"] = df["ds"].dt.dayofyear
    df["year"] = df["ds"].dt.year

    forecasts = []
    for target_date in target_dates:
        target_doy = target_date.dayofyear
        target_year = target_date.year
        mask = ((df["year"] >= target_year - years_back) & (df["year"] < target_year) & (np.abs(df["doy"] - target_doy) <= day_window))
        forecasts.append(df.loc[mask, "y"].mean())
    return pd.Series(forecasts, index=target_dates)

In [5]:
results = []
rs["ds"] = pd.to_datetime(rs["ds"])

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    
    train = rs.iloc[train_index].copy()
    test = rs.iloc[test_index].copy()
    
    # Target dates = the dates we want to forecast. There are 14 days.
    target_dates = test["ds"]
    
    # Seasonal forecast using only the training data (we will go back 5 years and take the average and use a day_window of 5 as well.)
    y_pred = seasonal_average_forecast(data=train, 
                                       target_dates=target_dates, 
                                       years_back=years_back_use,
                                       day_window=day_window_use)

    # We take the true values.
    y_true = test["y"].values
    
    # Compute the metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append the results of the metrics to the table as well as the fold number.
    results.append({"fold": i, "rmse": rmse, "mape": mape})

# Convert the data to a table for readability.
baseline_results_df = pd.DataFrame(results)

# We also include a new row which consists of the average RMSE and MAPE over each fold.
baseline_results_df.loc["mean"] = ["mean", baseline_results_df["rmse"].mean(), baseline_results_df["mape"].mean()]

baseline_results_df

,fold,rmse,mape
0,0,18.567526,0.251569
1,1,11.781678,0.202731
2,2,12.970780,0.219000
3,3,21.392505,0.237162
4,4,19.552544,0.180788
5,5,27.097225,0.257168
6,6,22.906809,0.217712
7,7,23.659416,0.240364
8,8,15.552538,0.210615
9,9,18.707209,0.207081


## Year Ago Rolling 4 Week Average 

In [6]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Just saving a copy for later
rs_saved = rs.copy()

In [7]:
# Tired of writing np.sqrt or typing a long name.
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = []

for fold, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    # Calculate the 4-week rolling average for the training data
    train_sorted = train.sort_values('ds') # making sure to sort it by date
    train_sorted['rolling_4w'] = train_sorted['y'].rolling(window=4, min_periods=1).mean()

    # This part of the code makes the predictions. We use the 'rolling_4w' column of the training set.
    y_pred = []
    y_true = test['y'].values

    for idx, row in test.iterrows():
        # Predict using the latest rolling average from the train data
        prediction = train_sorted['rolling_4w'].iloc[-1]  # Last value in the train rolling avg
        y_pred.append(prediction)
        
    # Calculate RMSE and MAPE for this fold
    fold_rmse = rmse(y_true, y_pred)
    fold_mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': fold, 'rmse': fold_rmse, 'mape': fold_mape})

rolling4w_results_df = pd.DataFrame(results)

overall_rmse = rolling4w_results_df['rmse'].mean()
overall_mape = rolling4w_results_df['mape'].mean()
rolling4w_results_df.loc['mean'] = ['mean', overall_rmse, overall_mape]
rolling4w_results_df

,fold,rmse,mape
0,0,15.558645,0.245443
1,1,20.800713,0.393455
2,2,14.232759,0.250932
3,3,19.439237,0.224473
4,4,16.407479,0.175315
5,5,21.056896,0.238259
6,6,28.874575,0.359895
7,7,28.131547,0.340850
8,8,17.986850,0.240132
9,9,25.218332,0.233300


## Results of the Two Baselines

In [8]:
models = {
    'baseline': baseline_results_df,
    'rolling4w': rolling4w_results_df,
}

all_results = []
for model_name, df in models.items():
    df['model'] = model_name
    all_results.append(df)

# Put all of the dataframes together into one dataframe for display
final_results_df = pd.concat(all_results, ignore_index=True)
# Make a pivot table so that we display rmse, mape and then each of the models and their results.
final_table = final_results_df.pivot(index='fold', columns='model', values=['rmse', 'mape'])
final_table.index = final_table.index.where(final_table.index != '-', 'mean')

final_table

rmse                 mape          
model   baseline  rolling4w  baseline rolling4w
fold                                           
0      18.567526  15.558645  0.251569  0.245443
1      11.781678  20.800713  0.202731  0.393455
2      12.970780  14.232759  0.219000  0.250932
3      21.392505  19.439237  0.237162  0.224473
4      19.552544  16.407479  0.180788  0.175315
5      27.097225  21.056896  0.257168  0.238259
6      22.906809  28.874575  0.217712  0.359895
7      23.659416  28.131547  0.240364  0.340850
8      15.552538  17.986850  0.210615  0.240132
9      18.707209  25.218332  0.207081  0.233300
10     24.272366  23.559006  0.191193  0.182605
11     18.153566  30.471151  0.225834  0.389008
12     24.853346  22.437214  0.249034  0.247137
13     16.828983  20.442209  0.179169  0.231404
14     22.776938  20.328332  0.239011  0.237294
15     18.884245  24.711731  0.208619  0.344475
16     14.104677  14.372282  0.225494  0.231666
17     12.975088  15.811106  0.209072  0.261849
18     12.866083  11.038488  0.175885  0.172352
19     12.724520  21.875459  0.348903  0.625055
20     13.934647  11.189520  0.359298  0.275160
21     15.114747  23.220796  0.588074  0.956630
22     19.059046  14.059821  0.505684  0.242077
23     12.462791  14.528298  0.280014  0.336888
24     13.023250  12.220811  0.313956  0.286328
25     12.671733  13.113515  0.278297  0.248402
mean   17.572856  19.272568  0.261605  0.306553